In [1]:
!pip install -q gradio transformers pillow torch torchvision sentencepiece gTTS
!pip install easyocr
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 14.8 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu118


In [2]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering
import gradio as gr
from gtts import gTTS
import os
import easyocr
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BLIP models
caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
caption_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)

# Initialize OCR
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [5]:
# OCR Extraction
def extract_text_from_image(image):
    result = reader.readtext(np.array(image))
    extracted_text = " ".join([text[1] for text in result])
    return extracted_text


# Description + ingredients
def generate_description(image):
    inputs = caption_processor(image, return_tensors="pt").to(device)
    output = caption_model.generate(**inputs, max_length=80)
    description = caption_processor.decode(output[0], skip_special_tokens=True)
    ingredients = ", ".join(description.replace(",", "").split()[:10])
    return description, ingredients


# Health analysis (Now uses OCR text primarily)
def health_analysis(description, ocr_text):
    lower_text = ocr_text.lower()

    unhealthy_keywords = ["sugar", "oil", "salt", "chocolate", "chips", "cream", "butter", "preservative"]

    if any(x in lower_text for x in unhealthy_keywords):
        type_food = "Processed / Junk"
        score = 40
        emoji = "❌"
    else:
        type_food = "Natural / Healthy"
        score = 85
        emoji = "🌱"

    advantages = "Provides energy" if "chocolate" in lower_text or "milk" in lower_text else "Rich in nutrients"
    disadvantages = "High sugar and calories may cause weight gain" if "sugar" in lower_text else "None significant"
    side_effects = "May affect blood sugar levels" if "sugar" in lower_text else "Rare"
    frequency = "Occasionally" if type_food=="Processed / Junk" else "Safe occasionally or daily in moderation"
    alternative = "Consider fresh fruit, nuts, or homemade snacks with minimal sugar" if type_food=="Processed / Junk" else "No replacement needed"

    return {
        "Type": type_food,
        "Score": score,
        "Emoji": emoji,
        "Advantages": advantages,
        "Disadvantages": disadvantages,
        "Side Effects": side_effects,
        "Frequency": frequency,
        "Better Alternative": alternative
    }


# VQA answer
def answer_question(image, question):
    if not question or question.strip()=="":
        return ""
    inputs = vqa_processor(image, question, return_tensors="pt").to(device)
    output = vqa_model.generate(**inputs, max_length=50)
    return vqa_processor.decode(output[0], skip_special_tokens=True)


# Comparison
def compare_products(info1, info2):
    comparison = f"Product 1: {info1['Emoji']} {info1['Type']}, Score: {info1['Score']}\n"
    comparison += f"Product 2: {info2['Emoji']} {info2['Type']}, Score: {info2['Score']}\n"

    if info1['Score'] > info2['Score']:
        comparison += "\n➡️ Product 1 is healthier than Product 2.\n"
    elif info2['Score'] > info1['Score']:
        comparison += "\n➡️ Product 2 is healthier than Product 1.\n"
    else:
        comparison += "\n➡️ Both products are similar in healthiness.\n"

    comparison += f"\nProduct 1 Side Effects: {info1['Side Effects']}\n"
    comparison += f"Product 2 Side Effects: {info2['Side Effects']}\n"

    comparison += f"\nSuggested Alternatives:\nProduct 1: {info1['Better Alternative']}\nProduct 2: {info2['Better Alternative']}\n"

    if info1['Score'] > info2['Score']:
        comparison += "\n🏆 Overall Verdict: Product 1 is healthier.\n"
    elif info2['Score'] > info1['Score']:
        comparison += "\n🏆 Overall Verdict: Product 2 is healthier.\n"
    else:
        comparison += "\n🏆 Overall Verdict: Both are similar.\n"

    return comparison


# Convert text to audio
def text_to_audio(text):
    tts = gTTS(text=text, lang='en')
    file_path = "/tmp/output.mp3"
    tts.save(file_path)
    return file_path

In [6]:
def food_vqa_app(image1, image2=None, question=None):
    results_text = ""

    if image1:
        desc1, ing1 = generate_description(image1)
        ocr_text1 = extract_text_from_image(image1)
        info1 = health_analysis(desc1, ocr_text1)

        res1 = f"📦 Product 1 Analysis:\n{desc1}\n"
        res1 += f"\n📝 Extracted Label Text:\n{ocr_text1}\n\n"

        for k,v in info1.items():
            if k!="Emoji":
                res1 += f"{k}: {v}\n"

        if question:
            ans1 = answer_question(image1, question)
            res1 += f"❓ Answer to your question: {ans1}\n"

        results_text += res1 + "\n" + "-"*60 + "\n"

    if image2:
        desc2, ing2 = generate_description(image2)
        ocr_text2 = extract_text_from_image(image2)
        info2 = health_analysis(desc2, ocr_text2)

        res2 = f"📦 Product 2 Analysis:\n{desc2}\n"
        res2 += f"\n📝 Extracted Label Text:\n{ocr_text2}\n\n"

        for k,v in info2.items():
            if k!="Emoji":
                res2 += f"{k}\n{v}\n"

        if question:
            ans2 = answer_question(image2, question)
            res2 += f"❓ Answer to your question: {ans2}\n"

        results_text += res2 + "\n" + "-"*60 + "\n"

    if image1 and image2:
        comparison = compare_products(info1, info2)
        results_text += "🔍 PRODUCT COMPARISON:\n" + comparison

    audio_file = text_to_audio(results_text)
    return results_text, audio_file

In [7]:
interface = gr.Interface(
    fn=food_vqa_app,
    inputs=[
        gr.Image(type="pil", label="Upload Food Image 1"),
        gr.Image(type="pil", label="Upload Food Image 2 (Optional)"),
        gr.Textbox(label="Ask a question (optional)", placeholder="e.g., Is it safe for kids?")
    ],
    outputs=[
        gr.Textbox(label="Analysis & Comparison", lines=20, placeholder="Results appear here...", interactive=False),
        gr.Audio(label="Listen to Analysis", type="filepath")
    ],
    title="🍎 Smart Food Product VQA & Health Comparison",
    description="Upload 1–2 food images. Get detailed analysis, health score, advantages/disadvantages, side effects, alternatives, smart tips, VQA answers, and listen to the output."
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://49045a3b40db9feeba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
